In [1]:
import numpy as np
import open3d as o3d
from pathlib import Path
import torch
import pickle
import os


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
UHM_DATASET_DIR = Path("C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train")


In [3]:
print(os.path.exists(UHM_DATASET_DIR))
print(os.path.isdir(UHM_DATASET_DIR))

True
True


In [4]:
DATASET_DIR = Path("C:/Eli Folder temp/geotransformer-faces-updated/data/faces")
os.makedirs(DATASET_DIR, exist_ok=True)

In [5]:
print(os.path.exists(DATASET_DIR))
print(os.path.isdir(DATASET_DIR))

True
True


In [6]:
all_files = list(UHM_DATASET_DIR.glob("*.ply"))

In [7]:
pca_data = torch.load("pca_basis_all.pth")
all_gt_z = pca_data['gt_z']
all_sorted_filenames = sorted([f.name for f in UHM_DATASET_DIR.glob("*.ply")])
name_to_idx = {name: idx for idx, name in enumerate(all_sorted_filenames)}
print(f"Loaded GT Z tensor of shape: {all_gt_z.shape}")

Loaded GT Z tensor of shape: torch.Size([32, 8000, 100])


In [8]:
RANDOM_SEED = 42

In [9]:
np.random.seed(RANDOM_SEED)

In [10]:
N_SAMPLES = 500

In [11]:
random_sample = np.random.choice(all_files, size=N_SAMPLES, replace=False)

In [12]:
random_sample

array([WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/3478.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/3890.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/2871.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/440.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/5867.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/4.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/2987.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/2267.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/5533.ply'),
       WindowsPath('C:/Eli Folder temp/ge

In [13]:
rng = np.random.default_rng(RANDOM_SEED)

In [14]:
def generate_random_view(
    pcd_points: np.ndarray, plane_width=0.001, plane_height_factor=1.2, plane_depth_factor=1.2, downsample_p=0.5, side_tol=1e-9
) -> np.ndarray:

    center = pcd_points.mean(axis=0)
    extent_y = np.ptp(pts[:, 1])
    extent_z = np.ptp(pts[:, 2])
    height = extent_y * plane_height_factor
    depth = extent_z * plane_depth_factor

    plane = o3d.geometry.TriangleMesh.create_box(width=plane_width, height=height, depth=depth)
    plane.translate([center[0] - plane_width / 2, center[1] - height / 2, center[2] - depth / 2])

    x_angle = rng.uniform(0, 2 * np.pi)
    y_angle = rng.uniform(0, 2 * np.pi)
    z_angle = rng.uniform(0, 2 * np.pi)
    R_plane = o3d.geometry.get_rotation_matrix_from_xyz([x_angle, y_angle, z_angle])
    plane.rotate(R_plane, center=center)

    initial_normal = np.array([1.0, 0.0, 0.0])
    rotated_normal = R_plane @ initial_normal
    rotated_normal /= np.linalg.norm(rotated_normal)

    vecs = pcd_points - center[np.newaxis, :]
    signed = vecs.dot(rotated_normal)
    side = rng.choice([1, -1])
    mask_side = (signed * side) > side_tol
    selected_pts = pcd_points[mask_side]
    orig_indices = np.nonzero(mask_side)[0]

    down_mask_bool = rng.choice([False, True], size=selected_pts.shape[0],
                                p=[1 - downsample_p, downsample_p])
    downsampled_pts = selected_pts[down_mask_bool]
    kept_indices = orig_indices[down_mask_bool]

    angles = rng.uniform(0, 2 * np.pi, size=3)
    R = o3d.geometry.get_rotation_matrix_from_xyz(angles)
    t = rng.uniform(-1.0, 1.0, size=3)
    transformed_points = (R @ downsampled_pts.T).T + t

    T = np.eye(4, dtype=float)
    T[:3, :3] = R
    T[:3, 3] = t

    R_inv = R.T
    t_inv = -R_inv @ t
    T_inv = np.eye(4, dtype=float)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv

    print(f"Reproyection error T_inv {np.linalg.norm((np.hstack((transformed_points, np.ones((transformed_points.shape[0], 1)))) @ T_inv.T)[:, :3] - pts[kept_indices])}")

    return transformed_points, R_inv, t_inv, plane, kept_indices

In [15]:
random_file = random_sample[0]

In [16]:
pcd = o3d.io.read_point_cloud(str(random_file))
pcd.colors = o3d.utility.Vector3dVector(np.ones((len(pcd.points), 3)) * 0.5)
pts = np.asarray(pcd.points)

In [17]:
view_points, view_r, view_t, view_plane, kept_indices = generate_random_view(pts)

Reproyection error T_inv 7.760227642219443e-15


In [18]:
view_r.dtype

dtype('float64')

In [19]:
view_pcd = o3d.geometry.PointCloud()
view_pcd.points = o3d.utility.Vector3dVector(view_points)

In [20]:
o3d.visualization.draw_plotly([view_pcd, view_plane, pcd])

In [21]:
T = np.eye(4, dtype=np.float64)
T[:3, :3] = view_r
T[:3, 3] = view_t

In [22]:
transformed_view = view_pcd.transform(T)

In [23]:
o3d.visualization.draw_plotly([transformed_view, view_plane])

In [24]:
def build_metadata_dict(scene_name, pcd0_path, pcd1_path, pcd_morphed_path, gt_z_path, R, t, frag_id1, kept_indices):
    metadata = {
        "overlap": 0,
        "pcd0": str(pcd0_path),
        "pcd1": str(pcd1_path),
        "pcd_morphed": str(pcd_morphed_path),
        "gt_z_path": str(gt_z_path),
        "rotation": R,
        "translation": t,
        "scene_name": scene_name,
        "frag_id0": 0,
        "frag_id1": frag_id1,
        "kept_indices": kept_indices,
    }
    return metadata


In [25]:
FULL_PC_NAME =  "full_face.pth"
MORPHED_PC_NAME = "full_morphed_face.pth"

print(f"Computing average face from {len(all_files)} files...")
all_pts_list = []

for f in all_files:
    temp_pcd = o3d.io.read_point_cloud(str(f))
    all_pts_list.append(np.asarray(temp_pcd.points))

average_pts = np.mean(np.stack(all_pts_list), axis=0)

print(f"Average point cloud created. Shape: {average_pts.shape}")

Computing average face from 8000 files...
Average point cloud created. Shape: (10788, 3)


In [26]:
def process_file(file_path: Path, n_views=10, folder="train", save=True):
    pcd = o3d.io.read_point_cloud(str(file_path))
    pts = np.asarray(pcd.points)

    print(f"Processing file: {file_path.stem} with {pts.shape[0]} points. Data type: {pts.dtype}")
    print("Reference: Global Average")

    subject_path = DATASET_DIR / "data" / folder / file_path.stem

    file_idx = name_to_idx[file_path.name]
    sample_gt_z = all_gt_z[:, file_idx, :] 

    if save:
        subject_path.mkdir(parents=True, exist_ok=True)
        torch.save(average_pts, subject_path / FULL_PC_NAME)
        torch.save(pts.astype(np.float32), subject_path / MORPHED_PC_NAME)
        torch.save(sample_gt_z, subject_path / "gt_z.pth")

    metadata_list = []

    for i in range(n_views):
        transformed_points, R_inv, t_inv, _, kept_indices = generate_random_view(pts)

        metadata = build_metadata_dict(
            scene_name=file_path.stem,
            pcd0_path=(Path(folder) / file_path.stem / FULL_PC_NAME).as_posix(),
            pcd1_path=(Path(folder) / file_path.stem / f"view_{i+1}.pth").as_posix(),
            pcd_morphed_path= (Path(folder) / file_path.stem / MORPHED_PC_NAME).as_posix(),
            gt_z_path=(Path(folder) / file_path.stem / "gt_z.pth").as_posix(),
            R=R_inv,
            t=t_inv,
            frag_id1=i + 1,
            kept_indices=kept_indices
        )
        metadata_list.append(metadata)
        if save:    
            torch.save(transformed_points, subject_path / f"view_{i+1}.pth")

    return metadata_list

In [27]:
train_size = int(np.round(0.8 * len(random_sample)))
val_size = int(np.round(0.1 * len(random_sample)))
test_size = len(random_sample) - train_size - val_size

In [28]:
train_size, val_size, test_size

(400, 50, 50)

In [29]:
train_metadata = []
for file_path in random_sample[:train_size]:
    metadata_list = process_file(file_path, n_views=10, folder="train")
    train_metadata.extend(metadata_list)
val_metadata = []
for file_path in random_sample[train_size:train_size+val_size]:
    metadata_list = process_file(file_path, n_views=10, folder="val")
    val_metadata.extend(metadata_list)

Processing file: 3478 with 10788 points. Data type: float64
Reference: Global Average
Reproyection error T_inv 1.115104214544172e-14
Reproyection error T_inv 7.833713128266641e-15
Reproyection error T_inv 7.28811155836796e-15
Reproyection error T_inv 1.1135677374336303e-14
Reproyection error T_inv 1.4099504587169224e-14
Reproyection error T_inv 9.420790493015863e-15
Reproyection error T_inv 8.274234566597533e-15
Reproyection error T_inv 6.595498045476598e-15
Reproyection error T_inv 8.576383040889647e-15
Reproyection error T_inv 8.963418094233676e-15
Processing file: 3890 with 10788 points. Data type: float64
Reference: Global Average
Reproyection error T_inv 2.8828044132959394
Reproyection error T_inv 2.882817493247143
Reproyection error T_inv 2.8293365698455015
Reproyection error T_inv 2.1994526196780844
Reproyection error T_inv 2.9777835032226223
Reproyection error T_inv 3.0101736598269246
Reproyection error T_inv 2.629919494170917
Reproyection error T_inv 2.664647947570063
Reproyec

In [30]:
test_metadata = []
for file_path in random_sample[train_size+val_size:]:
    metadata_list = process_file(file_path, n_views=1, folder="test")
    test_metadata.extend(metadata_list)

Processing file: 9882 with 10788 points. Data type: float64
Reference: Global Average
Reproyection error T_inv 4.910812027508777
Processing file: 3910 with 10788 points. Data type: float64
Reference: Global Average
Reproyection error T_inv 2.5122087814268474
Processing file: 4840 with 10788 points. Data type: float64
Reference: Global Average
Reproyection error T_inv 3.7103451196216577
Processing file: 3918 with 10788 points. Data type: float64
Reference: Global Average
Reproyection error T_inv 3.978004097499668
Processing file: 534 with 10788 points. Data type: float64
Reference: Global Average
Reproyection error T_inv 2.050404672328421
Processing file: 9846 with 10788 points. Data type: float64
Reference: Global Average
Reproyection error T_inv 3.131343729725499
Processing file: 8398 with 10788 points. Data type: float64
Reference: Global Average
Reproyection error T_inv 3.183427373009828
Processing file: 9002 with 10788 points. Data type: float64
Reference: Global Average
Reproyecti

In [31]:
METADATA_DIR = DATASET_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

In [32]:
with open(METADATA_DIR / "train.pkl", "wb") as f:
    pickle.dump(train_metadata, f)

In [33]:
with open(METADATA_DIR / "val.pkl", "wb") as f:
    pickle.dump(val_metadata, f)

In [34]:
demo_folder = DATASET_DIR / "demo"
demo_folder.mkdir(parents=True, exist_ok=True)

In [35]:
for ex_id, example in enumerate(test_metadata):
    example_ref = torch.load(DATASET_DIR / "data" / example["pcd0"], weights_only=False)
    example_src = torch.load(DATASET_DIR / "data" / example["pcd1"], weights_only=False)
    example_gt_morphed = torch.load(DATASET_DIR / "data" / example["pcd_morphed"], weights_only=False)
    np.save(demo_folder / f"ref_{ex_id}.npy", example_ref)
    np.save(demo_folder / f"src_{ex_id}.npy", example_src)
    np.save(demo_folder / f"morphed_full_{ex_id}.npy", example_gt_morphed)
    rot = example["rotation"]
    t = example["translation"]
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = rot
    T[:3, 3] = t
    np.save(demo_folder / f"gt_{ex_id}.npy", T)


### Inspect Test Data

In [36]:
#test_sample = test_metadata[np.random.randint(len(test_metadata))]
test_sample = test_metadata[0]

In [37]:
test_sample

{'overlap': 0,
 'pcd0': 'test/9882/full_face.pth',
 'pcd1': 'test/9882/view_1.pth',
 'pcd_morphed': 'test/9882/full_morphed_face.pth',
 'gt_z_path': 'test/9882/gt_z.pth',
 'rotation': array([[-0.00753022,  0.77386695, -0.63330344],
        [ 0.05030577,  0.63281265,  0.77266907],
        [ 0.99870547, -0.02604045, -0.04369519]]),
 'translation': array([ 0.01150157, -0.93106925,  0.62855211]),
 'scene_name': '9882',
 'frag_id0': 0,
 'frag_id1': 1,
 'kept_indices': array([    5,     8,    10, ..., 10773, 10774, 10775])}

In [38]:
test_src = torch.load(DATASET_DIR / "data" / test_sample["pcd1"], weights_only=False)
test_ref = torch.load(DATASET_DIR / "data" / test_sample["pcd0"], weights_only=False)

In [39]:
file_path = Path("C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train") / f"{test_sample['scene_name']}.ply" 

In [40]:
ref_original = pcd = o3d.io.read_point_cloud(str(file_path))

In [41]:
test_ref.shape

(10788, 3)

In [42]:
np.asarray(ref_original.points).shape

(10788, 3)

In [43]:
np.mean(np.linalg.norm(np.asarray(ref_original.points) - test_ref, axis=1))

np.float64(0.04229918946472553)

In [44]:
src_pcd = o3d.geometry.PointCloud()
src_pcd.points = o3d.utility.Vector3dVector(test_src)
src_pcd.paint_uniform_color([1.0, 0.0, 0.0])
ref_pcd = o3d.geometry.PointCloud()
ref_pcd.points = o3d.utility.Vector3dVector(test_ref)
ref_pcd.paint_uniform_color([0.0, 1.0, 0.0])


PointCloud with 10788 points.

In [45]:
o3d.visualization.draw_plotly([ref_pcd, src_pcd])

In [46]:
t = test_sample["translation"]
rot = test_sample["rotation"]
T = np.eye(4, dtype=np.float64)
T[:3, :3] = rot
T[:3, 3] = t

In [47]:
o3d.visualization.draw_plotly([ref_pcd, src_pcd.transform(T)])

In [48]:
kept_indices = test_sample["kept_indices"]

In [49]:
r_errors = (np.hstack((test_src, np.ones((test_src.shape[0], 1)))) @ T.T)[:, :3] - test_ref[kept_indices]

In [50]:
np.linalg.norm(r_errors, axis=1).max()

np.float64(0.10558711594803577)

In [51]:
print(f"Reproyection error T_inv {np.linalg.norm((np.hstack((test_src, np.ones((test_src.shape[0], 1)))) @ T.T)[:, :3] - test_ref[kept_indices])}")


Reproyection error T_inv 2.3464094686999206


In [52]:
error = np.asarray(ref_pcd.points)[kept_indices] - np.asarray(src_pcd.points)

In [53]:
np.linalg.norm(error, axis=1).mean()

np.float64(0.04178476687159769)

In [54]:
np.sum(np.linalg.norm(error, axis=1))

np.float64(101.78769209921197)